In [1]:
#!pip install easyocr

In [2]:
import cv2
import numpy as np
import easyocr
import pandas as pd


In [3]:

# # Initialize OCR reader
# reader = easyocr.Reader(['en'])

In [4]:
dataset_path = r'C:\Users\PRITAM\1.medic_project\dataset\raw_images'

In [5]:
import glob
import os

In [6]:
# import cv2
# import numpy as np
# import easyocr
# import pandas as pd
# import os
# import glob

# # Initialize OCR reader once (gpu=True if you have NVIDIA GPU)
# reader = easyocr.Reader(['en'], gpu=False)

# def analyze_medicine(image_path, actual_name):
#     # 1. Load Image
#     img = cv2.imread(image_path)
#     if img is None:
#         return None
    
#     # 2. Advanced Color Extraction (ROI based for better accuracy)
#     # Sirf center pixel ke bajaye center ka 20x20 area ka mean lete hain
#     h, w, _ = img.shape
#     roi = img[h//2-10:h//2+10, w//2-10:w//2+10]
#     avg_color_per_row = np.average(roi, axis=0)
#     avg_color = np.average(avg_color_per_row, axis=0)
    
#     # BGR to Gray mean for white detection
#     color_label = "White" if np.mean(avg_color) > 180 else "Colored"

#     # 3. Shape Detection (Using Aspect Ratio for higher fidelity)
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#     blurred = cv2.GaussianBlur(gray, (5, 5), 0)
#     edged = cv2.Canny(blurred, 30, 150)
#     contours, _ = cv2.findContours(edged, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
#     shape_label = "Unknown"
#     if contours:
#         c = max(contours, key=cv2.contourArea)
#         # Bounding box approach for Rectangular vs Round
#         x, y, w_rect, h_rect = cv2.boundingRect(c)
#         aspect_ratio = float(w_rect)/h_rect
        
#         # Contour approximation
#         peri = cv2.arcLength(c, True)
#         approx = cv2.approxPolyDP(c, 0.03 * peri, True)
        
#         if 0.9 <= aspect_ratio <= 1.1 and len(approx) > 6:
#             shape_label = "Round"
#         elif aspect_ratio > 1.2 or aspect_ratio < 0.8:
#             shape_label = "Capsule/Oval"
#         else:
#             shape_label = "Rectangular/Other"

#     # 4. Text Extraction (OCR)
#     # Image preprocessing for better OCR
#     gray_ocr = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#     results = reader.readtext(gray_ocr)
#     imprint_text = " ".join([str(res[1]) for res in results]).strip()

#     # 5. Structure the Data
#     data = {
#         "Medicine_Name": str(actual_name),
#         "Color": str(color_label),
#         "Shape": str(shape_label),
#         "Imprint": str(imprint_text),
#         "Image_Path": os.path.abspath(image_path)
#     }
    
#     return data

# # --- EXECUTION BLOCK ---
# # Path updated as per your system
# dataset_path = r'C:\Users\PRITAM\1.medic_project\dataset\raw_images'
# output_csv = "medicine_dataset.csv"

# image_files = []
# for ext in ['*.jpg', ['*.jpeg'], '*.png']:
#     image_files.extend(glob.glob(os.path.join(dataset_path, str(ext))))

# all_medicine_data = []

# print(f"Found {len(image_files)} images. Initializing Multi-modal Extraction...")

# for img_path in image_files:
#     file_name = os.path.basename(img_path).split('.')[0]
#     print(f"Processing: {file_name}...")
    
#     result = analyze_medicine(img_path, file_name)
    
#     if result:
#         all_medicine_data.append(result)

# # Saving to CSV (Appending mode for Continuous Learning)
# if all_medicine_data:
#     new_df = pd.DataFrame(all_medicine_data)
    
#     # Check if CSV exists to handle headers
#     if not os.path.isfile(output_csv):
#         new_df.to_csv(output_csv, index=False)
#     else:
#         new_df.to_csv(output_csv, mode='a', header=False, index=False)
        
#     print(f"\nSuccessfully updated {output_csv} with {len(all_medicine_data)} records.")
#     print(new_df[['Medicine_Name', 'Color', 'Shape', 'Imprint']])
# else:
#     print("Zero records processed. Check your image path.")

In [7]:
import sqlite3
import os
import hashlib
import cv2
import numpy as np
import easyocr
import pandas as pd
import glob
import shutil

# 1. Initialize OCR
reader = easyocr.Reader(['en'], gpu=False)

def get_image_hash(image_path):
    """Image ka unique fingerprint (Hash) nikalna."""
    hasher = hashlib.md5()
    with open(image_path, 'rb') as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()

def organize_and_save(image_path, manual_name):
    """Images ko project folder ke andar organize karke save karna."""
    target_dir = r'C:\Users\PRITAM\1.medic_project\dataset\processed_images'
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
    
    file_extension = os.path.splitext(image_path)[1]
    new_filename = f"{manual_name}_{get_image_hash(image_path)[:8]}{file_extension}"
    final_path = os.path.join(target_dir, new_filename)
    
    shutil.copy(image_path, final_path)
    return final_path

def analyze_medicine(image_path, actual_name):
    """Pill features (Color, Shape, Text) extract karna."""
    img = cv2.imread(image_path)
    if img is None: return None
    
    h, w, _ = img.shape
    roi = img[h//2-10:h//2+10, w//2-10:w//2+10]
    avg_color = np.average(np.average(roi, axis=0), axis=0)
    color_label = "White" if np.mean(avg_color) > 180 else "Colored"

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edged = cv2.Canny(blurred, 30, 150)
    contours, _ = cv2.findContours(edged, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    shape_label = "Unknown"
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w_rect, h_rect = cv2.boundingRect(c)
        aspect_ratio = float(w_rect)/h_rect
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.03 * peri, True)
        
        if 0.9 <= aspect_ratio <= 1.1 and len(approx) > 6: shape_label = "Round"
        elif aspect_ratio > 1.2 or aspect_ratio < 0.8: shape_label = "Capsule/Oval"
        else: shape_label = "Rectangular/Other"

    results = reader.readtext(gray)
    imprint_text = " ".join([str(res[1]) for res in results]).strip()

    return {
        "Medicine_Name": str(actual_name),
        "Color": str(color_label),
        "Shape": str(shape_label),
        "Imprint": str(imprint_text)
    }

def process_new_medicine(image_path, manual_name):
    """Main function: Duplicate check -> Analyze -> SQL Store."""
    conn = sqlite3.connect('medic_vault.db')
    cursor = conn.cursor()
    
    cursor.execute('''CREATE TABLE IF NOT EXISTS inventory 
                      (id INTEGER PRIMARY KEY AUTOINCREMENT,
                       name TEXT, color TEXT, shape TEXT, 
                       imprint TEXT, img_path TEXT, 
                       img_hash TEXT UNIQUE, 
                       is_trained INTEGER DEFAULT 0)''')
    
    current_hash = get_image_hash(image_path)
    cursor.execute("SELECT name FROM inventory WHERE img_hash = ?", (current_hash,))
    exists = cursor.fetchone()
    
    if exists:
        print(f"⚠️ SKIP: '{exists[0]}' already exists (Hash match).")
        conn.close()
        return

    data = analyze_medicine(image_path, manual_name)
    
    if data:
        final_path = organize_and_save(image_path, manual_name)
        try:
            cursor.execute("""INSERT INTO inventory 
                              (name, color, shape, imprint, img_path, img_hash) 
                              VALUES (?, ?, ?, ?, ?, ?)""", 
                           (data['Medicine_Name'], data['Color'], data['Shape'], 
                            data['Imprint'], final_path, current_hash))
            conn.commit()
            print(f"✅ SUCCESS: {manual_name} stored in Database!")
        except sqlite3.IntegrityError:
            print(f"🚨 Duplicate Error for {manual_name}")
            
    conn.close()

# --- EXECUTION ---
raw_images_dir = r'C:\Users\PRITAM\1.medic_project\dataset\raw_images'
all_files = []
for ext in ['*.jpg', '*.jpeg', '*.png']:
    all_files.extend(glob.glob(os.path.join(raw_images_dir, ext)))

print(f"Found {len(all_files)} images. Starting Database Pipeline...")

for img_p in all_files:
    med_name = os.path.basename(img_p).split('.')[0]
    process_new_medicine(img_p, med_name)

Using CPU. Note: This module is much faster with a GPU.


Found 5 images. Starting Database Pipeline...


C:\Users\PRITAM\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


✅ SUCCESS: citirizine stored in Database!
✅ SUCCESS: Azithromycin stored in Database!
✅ SUCCESS: metformin stored in Database!
✅ SUCCESS: Omeprazole stored in Database!
✅ SUCCESS: paraceta stored in Database!
